# Autoship Nudge Promo Incentive — Data Analysis (Subsequent Fix Order Rate, Demand-Event Definition)

This notebook builds up, CTE by CTE, the warehouse query used to establish the Subsequent Fix Order Rate baseline in `power_analysis_sfo_v1.ipynb`. Each step below adds one CTE and re-runs, so the final steps reproduce the exact queries used for sizing.

**Population:** Manual clients (i.e., not already enrolled in Autoship) who completed First Fix checkout with a Buy 1+ keep rate — the eligible population for the Autoship Nudge Promo Incentive Test, per the experiment's PRD.

**Source tables:**
- `curated.merch_sales_and_feedback`, an item-level Fix/direct-buy fact table, used to identify each client's First Fix and its keep rate.
- `curated.client_pulse_journal`, a daily client-state journal, used to read the subsequent-order signal itself: `cancellation_adjusted_last_fix_demand_ts`, the timestamp of a client's most recent Fix demand event (adjusted to exclude demand that was later cancelled), as of that journal row.

In [1]:
import pandas as pd
from amphibian import get_data_accessor

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

## Part A — Identifying each client's First Fix

`curated.merch_sales_and_feedback` is at the item grain (one row per item shipped), so a client's First Fix (`fix_number = 1`) spans multiple rows. This part collapses those rows to one row per client's first-Fix shipment, carrying forward the shipment's `autoship_or_manual` value and the count of items kept.

### A1 — raw item-level rows, `fix_number = 1`

In [2]:
query("""--sql
SELECT client_id, shipment_id, item_id, checkout_date, autoship_or_manual, sold_paid_fix_flag, business_line
FROM curated.merch_sales_and_feedback
WHERE fix_number = 1
  AND created_date >= DATE '2026-06-01'
ORDER BY client_id, shipment_id
LIMIT 8
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,shipment_id,item_id,checkout_date,autoship_or_manual,sold_paid_fix_flag,business_line
0,3008420,134765685,388218846,2026-07-05,autoship,0,Womens
1,3008420,134765685,387578192,2026-07-05,autoship,0,Womens
2,3008420,134765685,379180269,2026-07-05,autoship,0,Womens
3,3008420,134765685,372520741,2026-07-05,autoship,0,Womens
4,3008420,134765685,388424823,2026-07-05,autoship,0,Womens
5,3010612,134862228,381294500,2026-07-11,autoship,0,Womens
6,3010612,134862228,385837912,2026-07-11,autoship,0,Womens
7,3010612,134862228,375364919,2026-07-11,autoship,0,Womens


Each client's first-Fix shipment shows up as several rows (one per item). `autoship_or_manual` is constant within a shipment, so it collapses cleanly with `ARBITRARY()`; `sold_paid_fix_flag` is summed per shipment to get the number of items kept.

### A2 — collapse to one row per (client, shipment), and check cardinality

In [3]:
query("""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '2026-01-01'
    GROUP BY client_id, shipment_id
)
SELECT n_shipments, COUNT(*) AS n_clients
FROM (
    SELECT client_id, COUNT(DISTINCT shipment_id) AS n_shipments
    FROM first_fix
    GROUP BY client_id
)
GROUP BY n_shipments
ORDER BY n_shipments
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_shipments,n_clients
0,1,391727
1,2,542
2,3,17
3,4,7
4,6,1


The overwhelming majority of clients have exactly one shipment tagged `fix_number = 1`. The final query below keeps only the chronologically earliest such shipment per client via `ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date)`, so every eligible client contributes exactly one row.

## Part B — Manual vs. Autoship split, and why a recent cohort is used

`autoship_or_manual` marks whether a Fix was fulfilled under an active Autoship subscription at the time. Historically, clients could enroll in Autoship at signup, before ever receiving a Fix — so a meaningful share of *First* Fixes were historically already `'autoship'`. A more recent rollout moved Autoship enrollment to **after** First Fix checkout, once keep rate is known, via a post-checkout nudge. That shifts the manual share of First Fixes upward over time, which is why the baseline later in this notebook uses a recent reference month rather than a long historical average.

In [4]:
query("""--sql
SELECT
    DATE_TRUNC('month', created_date) AS month,
    autoship_or_manual,
    COUNT(DISTINCT client_id) AS n_clients
FROM curated.merch_sales_and_feedback
WHERE fix_number = 1
  AND created_date >= DATE '2025-08-01'
GROUP BY 1, 2
ORDER BY 1 DESC, 2
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,autoship_or_manual,n_clients
0,2026-08-01,autoship,5697
1,2026-08-01,manual,6541
2,2026-07-01,autoship,24579
3,2026-07-01,manual,28473
4,2026-06-01,autoship,20647
5,2026-06-01,manual,19868
6,2026-05-01,autoship,36903
7,2026-05-01,manual,12892
8,2026-04-01,autoship,42355
9,2026-04-01,manual,14193


## Part C — Keep rate: Buy 0 vs. Buy 1+

The PRD scopes this test to clients with a **Buy 1+** keep rate on their First Fix (kept at least one item) — Buy 0 clients always get the BAU Quick Fix experience with no Autoship nudge at all, and are out of scope for this promo/non-promo comparison. `sold_paid_fix_flag`, summed per shipment, gives the number of items kept.

In [5]:
query("""--sql
WITH first_fix AS (
    SELECT client_id, shipment_id, SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '2026-06-01'
    GROUP BY client_id, shipment_id
)
SELECT n_items_kept, COUNT(*) AS n_shipments
FROM first_fix
GROUP BY n_items_kept
ORDER BY n_items_kept
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_items_kept,n_shipments
0,0,43433
1,1,12266
2,2,12155
3,3,9740
4,4,4848
5,5,14443
6,6,1574
7,7,997
8,8,1967
9,9,245


Roughly half to three-fifths of recent First Fixes keep at least one item — the Buy 1+ gate this test's population applies.

## Part D — A fresh subsequent-order signal in `client_pulse_journal`

`curated.client_pulse_journal` is a slowly-changing-dimension journal: one row per day any tracked client attribute changes, bounded by `start_date`/`end_date`, ordered by `sequence`, with `is_current` marking the row that's still open. `cancellation_adjusted_last_fix_demand_ts` carries the timestamp of a client's most recent Fix demand event, already adjusted to exclude demand that was later cancelled.

The example below is a real client's full journal history, picked because their First Fix checkout (per Part A/C's population logic) falls on **2026-02-20**.

In [6]:
query("""--sql
SELECT client_id, sequence, start_date, end_date, is_current, cancellation_adjusted_last_fix_demand_ts, last_manual_shipment_checkout_ts
FROM curated.client_pulse_journal
WHERE client_id = 7140249
ORDER BY sequence
LIMIT 25
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,sequence,start_date,end_date,is_current,cancellation_adjusted_last_fix_demand_ts,last_manual_shipment_checkout_ts
0,7140249,1,2020-04-01 00:00:00.000,2026-02-12 00:00:00.000,0,None,None
1,7140249,2,2026-02-12 00:00:00.000,2026-02-17 00:00:00.000,0,2026-02-12 16:06:02.009,None
2,7140249,3,2026-02-17 00:00:00.000,2026-02-18 00:00:00.000,0,2026-02-12 16:06:02.009,None
3,7140249,4,2026-02-18 00:00:00.000,2026-02-19 00:00:00.000,0,2026-02-12 16:06:02.009,None
4,7140249,5,2026-02-19 00:00:00.000,2026-02-20 00:00:00.000,0,2026-02-12 16:06:02.009,2026-02-20 01:52:33.501
5,7140249,6,2026-02-20 00:00:00.000,2026-02-21 00:00:00.000,0,2026-02-12 16:06:02.009,2026-02-20 01:52:33.501
6,7140249,7,2026-02-21 00:00:00.000,2026-02-22 00:00:00.000,0,2026-02-12 16:06:02.009,2026-02-20 01:52:33.501
7,7140249,8,2026-02-22 00:00:00.000,2026-02-23 00:00:00.000,0,2026-02-22 16:38:43.165,2026-02-20 01:52:33.501
8,7140249,9,2026-02-23 00:00:00.000,2026-02-24 00:00:00.000,0,2026-02-22 16:38:43.165,2026-02-20 01:52:33.501
9,7140249,10,2026-02-24 00:00:00.000,2026-02-25 00:00:00.000,0,2026-02-22 16:38:43.165,2026-02-20 01:52:33.501


**Reading this:** this client already carries a `cancellation_adjusted_last_fix_demand_ts` from **2026-02-12** — 8 days *before* their First Fix checkout on 2026-02-20 (confirmed by `last_manual_shipment_checkout_ts` landing on that same date). That earlier timestamp predates the very Fix this test's nudge is shown after, so it must not be counted as a subsequent order. Then, at the row starting **2026-02-22** — 2 days after checkout — the timestamp updates to a genuinely fresh value, consistent with a new Fix being ordered after First Fix checkout. Requiring the demand timestamp to be strictly *after* First Fix checkout is what separates these two cases correctly; requiring only that the field be "populated" would have miscounted this client as having ordered a subsequent Fix 8 days before they even had their first one.

## Part E — A pre-existing timestamp that never refreshes

Not every client's demand timestamp updates after First Fix. The example below is another real client, whose First Fix checkout (confirmed by `last_manual_shipment_checkout_ts`) falls on **2026-02-06**.

In [7]:
query("""--sql
SELECT client_id, sequence, start_date, end_date, is_current, cancellation_adjusted_last_fix_demand_ts, last_manual_shipment_checkout_ts
FROM curated.client_pulse_journal
WHERE client_id = 3919729
ORDER BY sequence
LIMIT 25
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,sequence,start_date,end_date,is_current,cancellation_adjusted_last_fix_demand_ts,last_manual_shipment_checkout_ts
0,3919729,1,2020-04-01 00:00:00.000,2026-02-01 00:00:00.000,0,None,None
1,3919729,2,2026-02-01 00:00:00.000,2026-02-04 00:00:00.000,0,2026-02-01 09:33:15.264,None
2,3919729,3,2026-02-04 00:00:00.000,2026-02-05 00:00:00.000,0,2026-02-01 09:33:15.264,None
3,3919729,4,2026-02-05 00:00:00.000,2026-02-06 00:00:00.000,0,2026-02-01 09:33:15.264,2026-02-06 01:34:36.190
4,3919729,5,2026-02-06 00:00:00.000,2026-02-07 00:00:00.000,0,2026-02-01 09:33:15.264,2026-02-06 01:34:36.190
5,3919729,6,2026-02-07 00:00:00.000,2026-02-08 00:00:00.000,0,2026-02-01 09:33:15.264,2026-02-06 01:34:36.190
6,3919729,7,2026-02-08 00:00:00.000,2026-02-09 00:00:00.000,0,2026-02-01 09:33:15.264,2026-02-06 01:34:36.190
7,3919729,8,2026-02-09 00:00:00.000,2026-02-10 00:00:00.000,0,2026-02-01 09:33:15.264,2026-02-06 01:34:36.190
8,3919729,9,2026-02-10 00:00:00.000,2026-02-11 00:00:00.000,0,2026-02-01 09:33:15.264,2026-02-06 01:34:36.190
9,3919729,10,2026-02-11 00:00:00.000,2026-02-12 00:00:00.000,0,2026-02-01 09:33:15.264,2026-02-06 01:34:36.190


**Reading this:** `cancellation_adjusted_last_fix_demand_ts` is frozen at **2026-02-01** — 5 days *before* this client's First Fix checkout on 2026-02-06 — across every row shown, with no fresh value appearing afterward. A query that simply checked "is this field populated" would count this client as having ordered a subsequent Fix, when in fact the only demand on record predates their First Fix entirely. Requiring the timestamp to fall strictly after checkout excludes this case correctly.

## Part F — Quantifying the difference: full history vs. today's snapshot

To confirm the snapshot-vs-full-history distinction matters beyond individual examples, this step compares, across a real cohort, how many clients show a fresh post-First-Fix demand signal when reading only today's row versus scanning full history.

In [8]:
query("""--sql
WITH first_fix AS (
    SELECT client_id, shipment_id, MIN(checkout_date) AS checkout_date_1,
           ARBITRARY(autoship_or_manual) AS autoship_or_manual, SUM(sold_paid_fix_flag) AS n_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1 AND created_date >= DATE '2026-01-01' AND created_date < DATE '2026-03-01'
    GROUP BY client_id, shipment_id
),
eligible AS (
    SELECT client_id, checkout_date_1 FROM first_fix WHERE autoship_or_manual = 'manual' AND n_kept >= 1
),
pulse_current AS (
    SELECT client_id, cancellation_adjusted_last_fix_demand_ts
    FROM curated.client_pulse_journal
    WHERE is_current = '1'
),
pulse_full_history AS (
    SELECT e.client_id, MIN(p.cancellation_adjusted_last_fix_demand_ts) AS first_fresh_ts
    FROM eligible e
    JOIN curated.client_pulse_journal p
      ON p.client_id = e.client_id AND p.cancellation_adjusted_last_fix_demand_ts > e.checkout_date_1
    GROUP BY e.client_id
)
SELECT
    COUNT(*) AS n_eligible,
    SUM(CASE WHEN c.cancellation_adjusted_last_fix_demand_ts IS NOT NULL AND c.cancellation_adjusted_last_fix_demand_ts > e.checkout_date_1 THEN 1 ELSE 0 END) AS n_snapshot_only,
    (SELECT COUNT(*) FROM pulse_full_history) AS n_full_history
FROM eligible e
LEFT JOIN pulse_current c ON c.client_id = e.client_id
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_eligible,n_snapshot_only,n_full_history
0,18753,9539,9993


**Reading this:** the full-history read finds more subsequent orders than the snapshot-only read — consistent with Part E's frozen-timestamp pattern coexisting with genuine resets elsewhere in this population, not an isolated example. This notebook's baseline query (Part G) uses the full-history version throughout.

## Part G — The full baseline query

Putting it together: `eligible` applies the Manual + Buy 1+ gate (deduped to each client's earliest `fix_number = 1` shipment) and a 90-day maturation cutoff so the order-rate read has had time to resolve; `fresh_fix_demand` finds each eligible client's earliest post-First-Fix Fix demand timestamp, scanning full journal history per Parts D-F; the final `SELECT` flags `ordered_subsequent_fix` and aggregates to a monthly Subsequent Fix Order Rate and daily eligible volume. This is the exact query used in `power_analysis_sfo_v1.ipynb`.

In [9]:
MATURATION_DAYS = 90
COHORT_START = '2025-08-01'

baseline_query = f"""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '{COHORT_START}'
    GROUP BY client_id, shipment_id
),
eligible AS (
    SELECT client_id, checkout_date
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
      AND autoship_or_manual = 'manual'
      AND n_items_kept >= 1
      AND checkout_date <= CURRENT_DATE - INTERVAL '{MATURATION_DAYS}' DAY
),
fresh_fix_demand AS (
    SELECT e.client_id, MIN(p.cancellation_adjusted_last_fix_demand_ts) AS first_fresh_demand_ts
    FROM eligible e
    JOIN curated.client_pulse_journal p
      ON p.client_id = e.client_id
     AND p.cancellation_adjusted_last_fix_demand_ts > e.checkout_date
    GROUP BY e.client_id
),
joined AS (
    SELECT
        e.client_id,
        DATE_TRUNC('month', e.checkout_date) AS month,
        e.checkout_date,
        CASE WHEN f.first_fresh_demand_ts IS NOT NULL
              AND f.first_fresh_demand_ts <= e.checkout_date + INTERVAL '{MATURATION_DAYS}' DAY
             THEN 1 ELSE 0 END AS ordered_subsequent_fix
    FROM eligible e
    LEFT JOIN fresh_fix_demand f ON f.client_id = e.client_id
),
month_days AS (
    SELECT month, COUNT(DISTINCT checkout_date) AS days_observed
    FROM joined
    GROUP BY month
)
SELECT
    j.month,
    md.days_observed,
    COUNT(*) AS n_eligible,
    SUM(ordered_subsequent_fix) AS n_ordered,
    CAST(SUM(ordered_subsequent_fix) AS DOUBLE) / COUNT(*) AS subsequent_fix_order_rate,
    ROUND(COUNT(*) / CAST(md.days_observed AS DOUBLE), 1) AS eligible_per_day
FROM joined j
JOIN month_days md ON j.month = md.month
GROUP BY j.month, md.days_observed
ORDER BY j.month DESC
"""

query(baseline_query)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,n_eligible,n_ordered,subsequent_fix_order_rate,eligible_per_day
0,2026-05-01,8,2680,1451,0.541418,335.0
1,2026-04-01,30,10261,5380,0.524315,342.0
2,2026-03-01,31,10857,5494,0.506033,350.2
3,2026-02-01,28,8799,4599,0.522673,314.3
4,2026-01-01,31,10364,5274,0.508877,334.3
5,2025-12-01,31,8916,4746,0.532301,287.6
6,2025-11-01,30,7249,3369,0.464754,241.6
7,2025-10-01,31,9201,3992,0.433866,296.8
8,2025-09-01,30,9272,3948,0.425798,309.1
9,2025-08-01,31,7167,3125,0.436026,231.2


**Reference month:** the most recent fully-mature calendar month (all its days past the 90-day maturation cutoff) is the reference month `power_analysis_sfo_v1.ipynb` uses for its order-rate baseline.

## Part H — A fresher, decoupled daily-volume read

Daily eligible volume doesn't need the 90-day maturation wait the order-rate read needs — whether a client is Manual + Buy 1+ is known immediately at First Fix checkout. That means volume can be measured off the **most recent complete month**, even though that same month is too recent to supply a matured order-rate read.

In [10]:
query("""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '2026-05-01'
    GROUP BY client_id, shipment_id
),
eligible AS (
    SELECT client_id, checkout_date
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
      AND autoship_or_manual = 'manual'
      AND n_items_kept >= 1
),
month_days AS (
    SELECT DATE_TRUNC('month', checkout_date) AS month, COUNT(DISTINCT checkout_date) AS days_observed
    FROM eligible
    GROUP BY 1
)
SELECT
    DATE_TRUNC('month', e.checkout_date) AS month,
    md.days_observed,
    COUNT(*) AS n_eligible,
    ROUND(COUNT(*) / CAST(md.days_observed AS DOUBLE), 1) AS eligible_per_day
FROM eligible e
JOIN month_days md ON DATE_TRUNC('month', e.checkout_date) = md.month
GROUP BY DATE_TRUNC('month', e.checkout_date), md.days_observed
ORDER BY 1 DESC
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,n_eligible,eligible_per_day
0,2026-08-01,6,3318,553.0
1,2026-07-01,31,17747,572.5
2,2026-06-01,30,12151,405.0
3,2026-05-01,31,6821,220.0
4,2026-04-01,1,10,10.0


**Reading this:** the most recent complete month runs meaningfully higher than the matured reference month used for the rate — consistent with Part B's observation that the eligible population is still growing as a share of all First Fixes. `power_analysis_sfo_v1.ipynb` uses this more recent month's volume for duration calculations, decoupled from the rate, rather than understating the run rate with a stale volume figure.